# Proyek 6 — AI dan Citra Satelit untuk Pemantauan Deforestasi

## 1. Unduh dataset resmi ForestNet

In [7]:
import os, zipfile, urllib.request, pathlib, collections, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

DATA_URL="https://download.cs.stanford.edu/deep/ForestNetDataset.zip"
ARCHIVE=Path("ForestNetDataset.zip")

# Jika file ada tetapi ukurannya sangat kecil (misal < 100 KB), kemungkinan itu adalah halaman HTML error
if ARCHIVE.exists() and ARCHIVE.stat().st_size < 100000:
    print("Mendeteksi file rusak/tidak lengkap, menghapus...")
    ARCHIVE.unlink()

if not ARCHIVE.exists():
    print("Mengunduh dataset (ini mungkin memakan waktu beberapa menit)...")
    # Menggunakan opener dengan User-Agent agar tidak diblokir oleh server
    opener = urllib.request.build_opener()
    opener.addheaders = [('User-Agent', 'Mozilla/5.0')]
    urllib.request.install_opener(opener)
    urllib.request.urlretrieve(DATA_URL, ARCHIVE)

print("Ukuran arsip (GB):", round(ARCHIVE.stat().st_size/1e9,2))

try:
    with zipfile.ZipFile(ARCHIVE) as zf:
        names=zf.namelist()
        print("Jumlah file dalam arsip:", len(names))
        print("Contoh file:")
        print("\n".join(names[:40]))
        zf.extractall("forestnet_data")
except zipfile.BadZipFile:
    print("Error: File masih terdeteksi rusak. Silakan hapus file ForestNetDataset.zip secara manual di panel kiri dan jalankan ulang sel ini.")

ROOT=Path("forestnet_data")

Mengunduh dataset (ini mungkin memakan waktu beberapa menit)...
Ukuran arsip (GB): 3.36
Jumlah file dalam arsip: 50538
Contoh file:
deep/downloads/ForestNetDataset/
deep/downloads/ForestNetDataset/train.csv
deep/downloads/ForestNetDataset/test.csv
deep/downloads/ForestNetDataset/examples/
deep/downloads/ForestNetDataset/examples/0.908088103161242_127.95740046356693/
deep/downloads/ForestNetDataset/examples/0.908088103161242_127.95740046356693/auxiliary/
deep/downloads/ForestNetDataset/examples/0.908088103161242_127.95740046356693/auxiliary/peat.json
deep/downloads/ForestNetDataset/examples/0.908088103161242_127.95740046356693/auxiliary/gfc.npy
deep/downloads/ForestNetDataset/examples/0.908088103161242_127.95740046356693/auxiliary/srtm.npy
deep/downloads/ForestNetDataset/examples/0.908088103161242_127.95740046356693/auxiliary/osm.json
deep/downloads/ForestNetDataset/examples/0.908088103161242_127.95740046356693/auxiliary/ncep.npy
deep/downloads/ForestNetDataset/examples/0.90808810316124

## 2. Periksa label dan format dataset


In [8]:
csv_files=list(ROOT.rglob("*.csv"))
image_files=[]
for ext in ("*.png","*.jpg","*.jpeg","*.tif","*.tiff"):
    image_files.extend(ROOT.rglob(ext))
print("CSV:", [str(p) for p in csv_files[:20]])
print("Jumlah gambar:",len(image_files))
for p in csv_files[:5]:
    try:
        temp=pd.read_csv(p, nrows=5)
        print("\nCSV:",p,"\nKolom:",temp.columns.tolist())
        display(temp.head())
    except Exception as e: print("Tidak bisa membaca",p,e)

CSV: ['forestnet_data/deep/downloads/ForestNetDataset/val.csv', 'forestnet_data/deep/downloads/ForestNetDataset/test.csv', 'forestnet_data/deep/downloads/ForestNetDataset/train.csv']
Jumlah gambar: 10103

CSV: forestnet_data/deep/downloads/ForestNetDataset/val.csv 
Kolom: ['label', 'merged_label', 'latitude', 'longitude', 'year', 'example_path']


,label,merged_label,latitude,longitude,year,example_path
0,Other large-scale plantations,Plantation,-1.045932,114.226337,2008,examples/-1.0459323016426607_114.22633665467036
1,Grassland shrubland,Grassland shrubland,1.899183,100.592121,2013,examples/1.8991825699472704_100.59212073949948
2,Grassland shrubland,Grassland shrubland,-2.334839,114.303934,2016,examples/-2.334838886385819_114.3039338807062
3,Grassland shrubland,Grassland shrubland,0.148428,101.486320,2016,examples/0.1484277526675046_101.4863202521978
4,Oil palm plantation,Plantation,0.258476,101.121892,2006,examples/0.2584764324320208_101.12189177099783



CSV: forestnet_data/deep/downloads/ForestNetDataset/test.csv 
Kolom: ['label', 'merged_label', 'latitude', 'longitude', 'year', 'example_path']


,label,merged_label,latitude,longitude,year,example_path
0,Small-scale agriculture,Smallholder agriculture,1.446543,100.799778,2007,examples/1.446543071130837_100.79977801432716
1,Grassland shrubland,Grassland shrubland,-1.209593,122.040547,2016,examples/-1.2095927346679962_122.0405465386496
2,Timber plantation,Plantation,-0.205606,102.938756,2014,examples/-0.2056063310939432_102.93875617664573
3,Other,Other,-1.730708,113.711455,2012,examples/-1.7307080624887758_113.71145483864248
4,Oil palm plantation,Plantation,1.429489,100.823635,2010,examples/1.429488982681029_100.82363463176236



CSV: forestnet_data/deep/downloads/ForestNetDataset/train.csv 
Kolom: ['label', 'merged_label', 'latitude', 'longitude', 'year', 'example_path']


,label,merged_label,latitude,longitude,year,example_path
0,Timber plantation,Plantation,4.430849,96.101634,2014,examples/4.430849118860583_96.1016343478138
1,Other,Other,1.332341,109.374229,2012,examples/1.3323406178609702_109.37422873130464
2,Grassland shrubland,Grassland shrubland,-1.720266,115.006996,2016,examples/-1.720266384577504_115.00699582064485
3,Small-scale agriculture,Smallholder agriculture,-2.248346,104.135786,2011,examples/-2.248346072674411_104.1357857482906
4,Other large-scale plantations,Plantation,-2.100800,113.022376,2008,examples/-2.100800102991412_113.0223763234016


## 3. Pelatihan CNN baseline — langkah adaptasi
Dataset ForestNet pada halaman resmi menjelaskan empat label tingkat tinggi. Karena struktur berkas dapat berbeda, bagian berikut sengaja memvalidasi kolom label dan path gambar sebelum membangun model. Periksa output sel sebelumnya dan sesuaikan `LABEL_CSV`, `IMAGE_COL`, dan `LABEL_COL` jika diperlukan.

In [9]:
# Pilih CSV yang berisi label kategori dan nama/path gambar.
# Jalankan setelah melihat tabel kolom di atas; pilih file metadata yang sesuai.
label_candidates=[]
for p in csv_files:
    try:
        sample=pd.read_csv(p,nrows=3)
        cols=[str(c).lower() for c in sample.columns]
        if len(cols)>=2 and any("label" in c or "driver" in c or "class" in c for c in cols):
            label_candidates.append((p,sample.columns.tolist()))
    except Exception: pass
print("Kandidat CSV label:",label_candidates[:10])
print("Jika kandidat kosong, ikuti petunjuk penyesuaian pada README dan dokumentasi dataset resmi.")

Kandidat CSV label: [(PosixPath('forestnet_data/deep/downloads/ForestNetDataset/val.csv'), ['label', 'merged_label', 'latitude', 'longitude', 'year', 'example_path']), (PosixPath('forestnet_data/deep/downloads/ForestNetDataset/test.csv'), ['label', 'merged_label', 'latitude', 'longitude', 'year', 'example_path']), (PosixPath('forestnet_data/deep/downloads/ForestNetDataset/train.csv'), ['label', 'merged_label', 'latitude', 'longitude', 'year', 'example_path'])]
Jika kandidat kosong, ikuti petunjuk penyesuaian pada README dan dokumentasi dataset resmi.


## 4. Catatan model dan evaluasi
Untuk pelatihan CNN yang valid, data harus dipasangkan dengan label per gambar, lalu dibagi train/validation/test tanpa kebocoran lokasi/event. Gunakan confusion matrix, macro-F1, dan akurasi; perhatikan ketidakseimbangan kelas. Jangan menyatakan performa sebelum notebook benar-benar dijalankan dengan label yang terverifikasi.

## Sumber dan batasan
ForestNet merupakan dataset citra satelit Landsat 8 untuk Indonesia yang dikurasi dengan label penyebab deforestasi. Kelas yang tersedia adalah Plantation, Smallholder Agriculture, Grassland/shrubland, dan Other—tidak identik satu-per-satu dengan seluruh kategori pada resume webinar. CNN dapat membantu mengklasifikasikan pola citra, tetapi tidak membuktikan penyebab lapangan dan tetap membutuhkan validasi ahli/lapangan.